# Lab 9 (R) — Predict-It: a hypertension classifier on *your* cohort

Welcome to the data-science deep end — now in **R**. You're an analyst at a public-health unit, and you've been handed **your own slice of a real national health survey**: a personalized cohort drawn from the U.S. CDC's **NHANES** (National Health and Nutrition Examination Survey). No two students get the same people.

Your mission, in three acts:

1. **Meet your cohort** — load it, describe it.
2. **The correlate hunt** — find which health factor most tracks high blood pressure in *your* people.
3. **Build & tune a model** — train a classifier to *predict* hypertension, and do real **feature selection** to push its accuracy up.

Each person has: `age`, `female` (1/0), `bmi`, `waist`, `activity_min` (weekly active minutes), `smoker` (1/0), `sleep_hours`, `cholesterol`, and the thing we want to predict — `high_bp` (1 = high blood pressure, 0 = not).

> **Why R?** This is the R trial of Lab 9. Where Python leans on `pandas`, base R gives you data frames, `tapply()`, `cor()`, and — best of all — `glm()` for logistic regression *built in*. You write the analysis; R brings the statistics.

## Setup — run this cell, then leave it alone

This loads the helper tools and your personal cohort. You don't need to read the helper code — just run the next cell.

In [ ]:
# Helpers we provide for you (see lab9_helpers.R):
#   load_data()                    -> the full cleaned NHANES adult table
#   build_cohort(seed)             -> your personal cohort (everyone's is different)
#   evaluate_model(features, data) -> accuracy of a hypertension classifier
#   FEATURES                       -> the usable predictor column names
source("lab9_helpers.R")

cohort <- build_cohort(20240617)   # your personalized cohort
cat("Your cohort has", nrow(cohort), "adults.\n")
head(cohort)

## Part 1 — Meet your cohort

First, get a feel for your data. Run `summary(cohort)` to see the ranges and averages of every column.

In [ ]:
summary(cohort)

In [ ]:
summarize <- function(cohort) {
    # TODO: replace each NULL.
    #   n:                 how many people are in the cohort   (hint: nrow(cohort))
    #   mean_age:          the average age                     (hint: mean(cohort$age))
    #   hypertension_rate: the FRACTION with high_bp == 1      (hint: the mean of a 0/1 column IS the fraction of 1s)
    list(
        n = NULL,
        mean_age = NULL,
        hypertension_rate = NULL
    )
}

summarize(cohort)

Now compare groups. **Run** the cell below: it shows the hypertension rate for women vs. men, and across age bands. Do the patterns match what you'd expect?

In [ ]:
cat("Hypertension rate by sex (1 = female):\n")
print(tapply(cohort$high_bp, cohort$female, mean))

cat("\nHypertension rate by age band:\n")
age_band <- cut(cohort$age, breaks = c(18, 40, 60, 120), right = FALSE)
print(tapply(cohort$high_bp, age_band, mean))

## Part 2 — The correlate hunt

Which factor most closely tracks high blood pressure in *your* cohort? A **correlation** measures how strongly two columns move together: near **+1** (rise together), near **-1** (one rises as the other falls), near **0** (no relationship). We care about the *strength*, so we look at the **absolute value**.

**Run** the cell below to see each feature's correlation with `high_bp`, sorted strongest-first.

In [ ]:
cors <- vapply(FEATURES, function(f) cor(cohort[[f]], cohort$high_bp), numeric(1))
sort(abs(cors), decreasing = TRUE)

In [ ]:
strongest_predictor <- function(cohort) {
    cors <- vapply(FEATURES, function(f) cor(cohort[[f]], cohort$high_bp), numeric(1))
    # TODO: return the feature NAME with the largest ABSOLUTE correlation.
    #   hint: abs(cors) gives sizes; which.max() gives its position; names() gives the label.
    NULL
}

strongest_predictor(cohort)

A picture makes it obvious. **Run** this to chart the strength of each correlation:

In [ ]:
barplot(sort(abs(cors)),
        horiz = TRUE, las = 1,
        xlab = "|correlation| with high blood pressure",
        main = "What tracks hypertension in your cohort?")

## Part 3 — Build & tune a model

Knowing what *correlates* with hypertension is good. **Predicting** it is better. In this part you'll build a model that guesses whether a person has high blood pressure — and then make it better.

### What's a classifier?

A **classifier** looks at a person's features (`age`, `bmi`, `waist`, ...) and guesses a **yes/no** answer — here, *does this person have high blood pressure?* We don't write that rule by hand; we **train** it on people whose answer we already know.

To know it's any good, we hold back some people as a **test set** and measure **accuracy**: the fraction of those *unseen* people the model gets right. **1.0** is perfect; **0.5** is a coin flip.

`evaluate_model(features, data)` does all of this for you (it uses R's built-in `glm()`). Your one job is to choose **which features** to hand it. **Run** the cell to see how a model using *every* feature does:

In [ ]:
evaluate_model(FEATURES, cohort)

Now the real skill: **feature selection**. More features is *not* always better — useless ones add noise. Edit `select_features()` to return the columns you want, then run the cell to see your accuracy.

**Start** with the two or three features that had the strongest correlations in Part 2; run it, then experiment.

**[GOAL] Goal: get your accuracy to at least 0.70.** (Hint: which features had the strongest correlations? Do the weak ones — `sleep_hours`, `smoker` — actually help?)

In [ ]:
select_features <- function() {
    # TODO: this is just a guess! Which features actually predict high blood pressure?
    c("sleep_hours")
}

my_features <- select_features()
cat("Features:", paste(my_features, collapse = ", "), "\n")
cat(sprintf("Accuracy on your cohort: %.3f\n", evaluate_model(my_features, cohort)))

## Reflection

In a sentence or two each (just type into this cell):

1. Which feature was the strongest predictor of hypertension in your cohort? Did that surprise you?
2. What was the smallest set of features that still reached your best accuracy?
3. Your model is right about 3 out of 4 times. Why might that *not* be good enough to use on real patients?